# Temporal Downsampling Ablation — Baseline CNN

Investigates how CTC performance degrades as the EMG sample rate is reduced by integer factors
via the `TemporalDownsample(factor)` transform (anti-aliased resampling before `LogSpectrogram`).
`in_features` is unchanged across all conditions.

| Sample rate | Factor | Val CER | Test CER |
|---|---|---|---|
| 2000 Hz (baseline) | 1× | **18.52%** | 22.28% |
| 1000 Hz | 2× | 52.53% | 38.38% |
| 500 Hz | 4× | 58.42% | 46.57% |
| 250 Hz | 8× | 79.15% | 76.12% |
| 125 Hz | 16× | 99.41% | 99.98% |

```bash
source /home/benforbes/emg2qwerty/venv/bin/activate
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

## 1. Setup & Patch

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / 'emg2qwerty').is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PLAYGROUND = REPO_ROOT / 'Playground_Ben'
UCLA = {'blue': '#2774AE', 'gold': '#FFD100', 'dark_blue': '#003B5C', 'dark_gold': '#FFB300'}

# Patch emg2qwerty (TemporalDownsample transform lives in transforms.py)
shutil.copy(PLAYGROUND / 'emg2qwerty/transforms.py', REPO_ROOT / 'emg2qwerty/transforms.py')
shutil.copy(PLAYGROUND / 'emg2qwerty/lightning.py',  REPO_ROOT / 'emg2qwerty/lightning.py')

# Copy temporal downsample configs
for cfg in (PLAYGROUND / 'config/transforms').glob('temporal_downsample_*.yaml'):
    shutil.copy(cfg, REPO_ROOT / 'config/transforms' / cfg.name)

print('Patched and configs copied.')
print(f'Repo root: {REPO_ROOT}')

## 2. Train All Temporal Conditions

Run in sequence. Skip any condition already trained.

### 2.1 — 2000 Hz baseline (no override)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'2000 Hz exit code: {result.returncode}')

### 2.2 — 1000 Hz (factor=2)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'transforms=temporal_downsample_2', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'1000 Hz exit code: {result.returncode}')

### 2.3 — 500 Hz (factor=4)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'transforms=temporal_downsample_4', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'500 Hz exit code: {result.returncode}')

### 2.4 — 250 Hz (factor=8)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'transforms=temporal_downsample_8', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'250 Hz exit code: {result.returncode}')

### 2.5 — 125 Hz (factor=16)

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'transforms=temporal_downsample_16', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'125 Hz exit code: {result.returncode}')

## 3. Results from Pre-computed CSVs

In [ ]:
df = pd.read_csv(PLAYGROUND / 'results/sampling_ablation_table.csv').sort_values('sample_rate_hz', ascending=False)
print(df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Temporal Downsampling Ablation — Baseline TDSConvCTC', fontsize=13)

tick_labels = [f"{int(hz)} Hz" for hz in df['sample_rate_hz']]
x = range(len(df))

axes[0].plot(list(x), df['val_cer_pct'].values,  'o-', color=UCLA['blue'],      lw=2.5, ms=8, label='Val CER')
axes[0].plot(list(x), df['test_cer_pct'].values, 's--', color=UCLA['dark_gold'], lw=2,   ms=7, label='Test CER')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(tick_labels)
axes[0].set(xlabel='Sample rate (high → low)', ylabel='CER (%)', title='CER vs Sample Rate')
axes[0].set_ylim(0, 110); axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

axes[1].barh(tick_labels, df['training_time_sec'].values / 3600,
             color=UCLA['blue'], edgecolor='white')
axes[1].set(xlabel='Training time (hrs)', ylabel='Sample rate', title='Training Time vs Sample Rate')
axes[1].grid(True, alpha=0.25, ls='--', axis='x')

plt.tight_layout()
plt.savefig(PLAYGROUND / 'plots/temporal_ablation_final.png', dpi=150)
plt.show()